# Notebook 10: k-Nearest Neighbors & Instance-Based Learning
### Part 10/30 – ML Mastery Series for Python Experts

## KNN – The Simplest Yet Surprisingly Powerful Classifier

You already know eager learners like trees and SVM — now meet the laziest learner of them all: one that doesn't even try until you ask it a question. Here's what makes KNN unique:

- **No training phase = lazy learner**: KNN stores the training data and defers all computation until prediction time—there's no model to "fit" in the traditional sense
- **Memorizes training data**: The "model" is literally the dataset itself; every prediction requires consulting the stored instances
- **Makes predictions by majority vote / averaging neighbors**: Classification uses plurality voting, regression uses mean/median of neighbor targets
- **Very intuitive**: The algorithm embodies the principle that similar things exist in close proximity—if it walks like a duck and quacks like a duck, it's probably a duck
- **But slow at prediction time**: Query time grows with training set size (often $O(n \log n)$ with efficient trees, $O(n)$ naive), making real-time inference expensive
- **Sensitive to irrelevant features & scale**: Distance calculations are dominated by features with large ranges; one irrelevant high-variance feature can ruin the neighborhood structure

## Learning Objectives

By the end of this notebook, you will:

- **Understand Euclidean/Manhattan/Minkowski distances**: Know when each metric is appropriate and how $p$ affects the geometry of neighborhoods
- **Choose optimal k via CV**: Navigate the bias-variance trade-off by selecting k that minimizes generalization error
- **Visualize Voronoi-like decision boundaries**: See how decision regions form around training points and how k smooths them
- **Grasp curse of dimensionality**: Understand why "nearest" becomes meaningless as dimensions explode
- **Use distance weighting**: Give closer neighbors more influence than distant ones
- **Compare uniform vs distance weights**: Know when inverse-distance weighting improves predictions
- **Handle regression with KNNRegressor**: Apply instance-based learning to continuous targets
- **Know when KNN shines vs when it fails**: Recognize the sweet spot (low dimensions, medium data, local structure) vs failure modes (high dimensions, huge datasets, global patterns)

## 📍 1. Basic KNN Classification – Iris Dataset

Let's start with the classic Iris dataset. Notice how we use a Pipeline—scaling is absolutely critical for KNN since it relies on distance calculations.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, confusion_matrix

# Set plotting style
sns.set_theme()
%matplotlib inline

# Load iris dataset
iris = load_iris()
X, y = iris.data, iris.target

# Split with stratification to maintain class balance
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print(f"Training samples: {X_train.shape[0]}")
print(f"Test samples: {X_test.shape[0]}")
print(f"Features: {X_train.shape[1]} ({iris.feature_names})")

In [ ]:
# Build pipeline: scaling is non-negotiable for KNN
knn_pipeline = Pipeline([
    ('scaler', StandardScaler()),  # Zero mean, unit variance—essential for distance metrics
    ('knn', KNeighborsClassifier(n_neighbors=5))  # Default k=5, uniform weights
])

# Fit the pipeline (scaler learns params, KNN just stores data)
knn_pipeline.fit(X_train, y_train)

# Predictions
y_pred = knn_pipeline.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"Test accuracy with k=5: {accuracy:.4f}")

# Confusion matrix visualization
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=iris.target_names,
            yticklabels=iris.target_names)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('KNN Confusion Matrix (k=5)')
plt.show()

# Show which classes get confused
print("\nClass-wise accuracy:")
for i, name in enumerate(iris.target_names):
    mask = y_test == i
    class_acc = accuracy_score(y_test[mask], y_pred[mask])
    print(f"  {name}: {class_acc:.3f}")

## 📍 2. Visualizing Decision Boundaries for Different k

Now let's visualize how k controls model complexity. We'll use only two features (petal length and width) so we can plot the decision surface.

Watch how:
- **k=1**: Overfitting (jagged boundaries that capture noise)
- **k=15**: Good balance (smooth but accurate)
- **k=50**: Underfitting (overly smooth, misses local structure)

In [ ]:
# Use only petal features for 2D visualization
X_2d = X_train[:, [2, 3]]  # Petal length, petal width
y_train_2d = y_train
X_test_2d = X_test[:, [2, 3]]

# Different k values to compare
k_values = [1, 5, 15, 50]
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.ravel()

for idx, k in enumerate(k_values):
    # Fit KNN on 2D data (no pipeline needed since features are comparable)
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_2d, y_train_2d)
    
    # Create mesh for decision surface
    h = 0.02
    x_min, x_max = X_2d[:, 0].min() - 0.5, X_2d[:, 0].max() + 0.5
    y_min, y_max = X_2d[:, 1].min() - 0.5, X_2d[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))
    
    # Predict on mesh
    Z = knn.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    
    # Plot decision boundary
    axes[idx].contourf(xx, yy, Z, alpha=0.4, cmap='viridis')
    
    # Plot training points
    scatter = axes[idx].scatter(X_2d[:, 0], X_2d[:, 1], c=y_train_2d, 
                               cmap='viridis', edgecolors='k', s=50)
    
    # Calculate accuracy on test set
    acc = knn.score(X_test_2d, y_test)
    
    axes[idx].set_xlabel('Petal Length')
    axes[idx].set_ylabel('Petal Width')
    axes[idx].set_title(f'k = {k}\nTest Accuracy: {acc:.3f}')
    axes[idx].set_xlim(x_min, x_max)
    axes[idx].set_ylim(y_min, y_max)

plt.suptitle('KNN Decision Boundaries: Effect of k', fontsize=16, y=0.98)
plt.tight_layout()
plt.show()

print("Key observation: k=1 creates isolated 'islands' (overfitting).")
print("k=50 smooths too much, merging distinct regions (underfitting).")

## 📍 3. Choosing k – Cross-Validation & Elbow Method

How do we pick the optimal k? The elbow method uses cross-validation to find where increasing k stops helping. We look for the "elbow" in the accuracy curve—beyond that, we're just adding bias without reducing variance.

In [ ]:
from sklearn.model_selection import cross_val_score, GridSearchCV
from sklearn.datasets import load_wine

# Use wine dataset for richer evaluation
wine = load_wine()
X_wine, y_wine = wine.data, wine.target

# Test odd k values from 1 to 50 (odd to avoid ties in binary case)
k_range = range(1, 51, 2)
cv_scores = []
cv_stds = []

for k in k_range:
    knn = Pipeline([
        ('scaler', StandardScaler()),
        ('knn', KNeighborsClassifier(n_neighbors=k))
    ])
    
    # 10-fold CV for robust estimate
    scores = cross_val_score(knn, X_wine, y_wine, cv=10, scoring='accuracy')
    cv_scores.append(scores.mean())
    cv_stds.append(scores.std())

# Plot elbow curve
plt.figure(figsize=(12, 6))
plt.plot(k_range, cv_scores, 'b-o', linewidth=2, markersize=6)
plt.fill_between(k_range, 
                 np.array(cv_scores) - np.array(cv_stds),
                 np.array(cv_scores) + np.array(cv_stds),
                 alpha=0.2)
plt.xlabel('k (Number of Neighbors)')
plt.ylabel('Cross-Validation Accuracy')
plt.title('Elbow Method for Selecting k\n(Shaded area = ±1 std)')
plt.grid(True, alpha=0.3)
plt.xticks(range(1, 51, 4))

# Find optimal k
optimal_k = list(k_range)[np.argmax(cv_scores)]
plt.axvline(x=optimal_k, color='r', linestyle='--', 
            label=f'Optimal k = {optimal_k}')
plt.legend()
plt.show()

print(f"Optimal k from CV: {optimal_k}")
print(f"Best CV accuracy: {max(cv_scores):.4f}")

In [ ]:
# GridSearchCV for comprehensive tuning
param_grid = {
    'knn__n_neighbors': range(1, 31, 2),
    'knn__weights': ['uniform', 'distance'],
    'knn__metric': ['euclidean', 'manhattan']
}

grid_search = GridSearchCV(
    Pipeline([('scaler', StandardScaler()), 
              ('knn', KNeighborsClassifier())]),
    param_grid,
    cv=10,
    scoring='accuracy',
    n_jobs=-1
)

grid_search.fit(X_wine, y_wine)

print(f"Best parameters: {grid_search.best_params_}")
print(f"Best CV accuracy: {grid_search.best_score_:.4f}")

# Show top 5 configurations
results_df = pd.DataFrame(grid_search.cv_results_)
top5 = results_df.nlargest(5, 'mean_test_score')[
    ['param_knn__n_neighbors', 'param_knn__weights', 
     'param_knn__metric', 'mean_test_score']
]
print("\nTop 5 configurations:")
print(top5.to_string(index=False))

## 📍 4. Distance Metrics & Weighting Schemes

KNN is fundamentally about distance. The metric you choose changes what "nearest" means:

- **Euclidean** ($L_2$): $\sqrt{\sum(x_i - y_i)^2}$ — straight-line distance, good for dense continuous data
- **Manhattan** ($L_1$): $\sum|x_i - y_i|$ — grid-like distance, robust to outliers, good for high dimensions
- **Minkowski** ($L_p$): $(\sum|x_i - y_i|^p)^{1/p}$ — generalization where $p=2$ is Euclidean, $p=1$ is Manhattan

Weighting schemes:
- **uniform**: All k neighbors vote equally
- **distance**: Weight by $1/d$ — closer neighbors have more say

In [ ]:
from sklearn.datasets import make_moons

# Generate non-linear data
X_moons, y_moons = make_moons(n_samples=300, noise=0.3, random_state=42)

# Compare metrics and weights
configs = [
    ('euclidean', 'uniform'),
    ('manhattan', 'uniform'),
    ('euclidean', 'distance'),
    ('manhattan', 'distance')
]

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.ravel()

for idx, (metric, weights) in enumerate(configs):
    # Create pipeline
    knn = Pipeline([
        ('scaler', StandardScaler()),
        ('knn', KNeighborsClassifier(n_neighbors=15, metric=metric, weights=weights))
    ])
    knn.fit(X_moons, y_moons)
    
    # Create mesh
    h = 0.02
    x_min, x_max = X_moons[:, 0].min() - 0.5, X_moons[:, 0].max() + 0.5
    y_min, y_max = X_moons[:, 1].min() - 0.5, X_moons[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))
    
    Z = knn.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    
    # Plot
    axes[idx].contourf(xx, yy, Z, alpha=0.4, cmap='coolwarm')
    axes[idx].scatter(X_moons[:, 0], X_moons[:, 1], c=y_moons, 
                     cmap='coolwarm', edgecolors='k', s=40)
    
    acc = knn.score(X_moons, y_moons)
    axes[idx].set_title(f'Metric: {metric}, Weights: {weights}\nAccuracy: {acc:.3f}')

plt.suptitle('Impact of Distance Metric and Weighting', fontsize=16, y=0.98)
plt.tight_layout()
plt.show()

print("Observation: Distance weighting often smooths boundaries.")
print("Manhattan vs Euclidean changes which points are considered 'near'.")

## 📍 5. Curse of Dimensionality – Demonstration

The curse of dimensionality is KNN's nemesis. As dimensions increase:
1. Distance between any two points grows
2. Relative contrast between near and far points vanishes
3. All points become equidistant
4. "Nearest neighbor" becomes a random choice

Mathematically, the volume of space grows exponentially, requiring exponentially more data to maintain density.

In [ ]:
from sklearn.datasets import make_classification
from sklearn.model_selection import cross_val_score

# Test dimensions: 2, 5, 10, 20, 50, 100
dimensions = [2, 5, 10, 20, 50, 100]
cv_means = []
cv_stds = []

n_samples = 500  # Fixed sample count

for n_features in dimensions:
    # Generate data with increasing dimensions (2 informative, rest noise)
    X, y = make_classification(
        n_samples=n_samples,
        n_features=n_features,
        n_informative=2,  # Only 2 features actually matter!
        n_redundant=0,
        n_clusters_per_class=1,
        random_state=42
    )
    
    knn = Pipeline([
        ('scaler', StandardScaler()),
        ('knn', KNeighborsClassifier(n_neighbors=5))
    ])
    
    scores = cross_val_score(knn, X, y, cv=5, scoring='accuracy')
    cv_means.append(scores.mean())
    cv_stds.append(scores.std())
    
    print(f"Dims: {n_features:3d} | Accuracy: {scores.mean():.3f} (+/- {scores.std():.3f})")

# Plot the curse
plt.figure(figsize=(10, 6))
plt.semilogx(dimensions, cv_means, 'ro-', linewidth=2, markersize=8)
plt.fill_between(dimensions, 
                 np.array(cv_means) - np.array(cv_stds),
                 np.array(cv_means) + np.array(cv_stds),
                 alpha=0.2)
plt.xlabel('Number of Features (Dimensions)')
plt.ylabel('Cross-Validation Accuracy')
plt.title('The Curse of Dimensionality\n(Fixed n_samples=500, only 2 informative features)')
plt.grid(True, alpha=0.3)
plt.xticks(dimensions, [str(d) for d in dimensions])
plt.show()

print("\nKey insight: Even with scaling, KNN degrades as irrelevant dimensions drown out signal.")

In [ ]:
# Demonstrate with purely random (noisy) features added
np.random.seed(42)
X_base, y = make_classification(n_samples=500, n_features=2, n_informative=2, 
                                n_redundant=0, random_state=42)

noise_levels = [0, 5, 10, 20, 50, 100]  # Number of random noise features to add
noise_results = []

for n_noise in noise_levels:
    if n_noise == 0:
        X = X_base
    else:
        # Add random noise features
        noise = np.random.randn(500, n_noise)
        X = np.hstack([X_base, noise])
    
    knn = Pipeline([
        ('scaler', StandardScaler()),
        ('knn', KNeighborsClassifier(n_neighbors=5))
    ])
    
    scores = cross_val_score(knn, X, y, cv=5)
    noise_results.append({'n_noise_features': n_noise, 'accuracy': scores.mean()})

noise_df = pd.DataFrame(noise_results)
plt.figure(figsize=(10, 6))
plt.plot(noise_df['n_noise_features'], noise_df['accuracy'], 'g-s', linewidth=2)
plt.xlabel('Number of Random Noise Features Added')
plt.ylabel('CV Accuracy')
plt.title('Performance Degradation from Irrelevant Features')
plt.grid(True, alpha=0.3)
plt.show()

print("This is why feature selection matters before applying KNN!")

## 📍 6. KNN for Regression – Quick Example

KNN isn't just for classification. For regression, it averages (or median-izes) the target values of the k nearest neighbors. This makes it a **local averaging** method—essentially a kernel smoother with uniform or distance-based kernel.

In [ ]:
from sklearn.datasets import load_diabetes
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

# Load diabetes dataset
diabetes = load_diabetes()
X, y = diabetes.data, diabetes.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

# Compare three regressors
models = {
    'KNN (k=5)': Pipeline([
        ('scaler', StandardScaler()),
        ('knn', KNeighborsRegressor(n_neighbors=5))
    ]),
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42)
}

results = {}
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for idx, (name, model) in enumerate(models.items()):
    # Fit and predict
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    # Metrics
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    results[name] = {'MSE': mse, 'R2': r2}
    
    # Predicted vs Actual plot
    axes[idx].scatter(y_test, y_pred, alpha=0.6, edgecolors='k')
    axes[idx].plot([y_test.min(), y_test.max()], 
                   [y_test.min(), y_test.max()], 'r--', lw=2)
    axes[idx].set_xlabel('Actual')
    axes[idx].set_ylabel('Predicted')
    axes[idx].set_title(f'{name}\nMSE: {mse:.1f}, R²: {r2:.3f}')
    axes[idx].grid(True, alpha=0.3)

plt.suptitle('KNN Regression vs Baselines', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# Summary table
results_df = pd.DataFrame(results).T
print("\nRegression Performance Comparison:")
print(results_df.round(3))

In [ ]:
# Residual analysis for KNN
knn_reg = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsRegressor(n_neighbors=5))
)
knn_reg.fit(X_train, y_train)
y_pred_knn = knn_reg.predict(X_test)
residuals = y_test - y_pred_knn

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Residual vs Predicted
ax1.scatter(y_pred_knn, residuals, alpha=0.6, edgecolors='k')
ax1.axhline(y=0, color='r', linestyle='--')
ax1.set_xlabel('Predicted Values')
ax1.set_ylabel('Residuals')
ax1.set_title('Residual Plot: KNN Regression')
ax1.grid(True, alpha=0.3)

# Distribution of residuals
ax2.hist(residuals, bins=20, edgecolor='k', alpha=0.7)
ax2.set_xlabel('Residual Value')
ax2.set_ylabel('Frequency')
ax2.set_title('Distribution of Residuals')
ax2.axvline(x=0, color='r', linestyle='--')

plt.tight_layout()
plt.show()

print(f"Residual mean: {residuals.mean():.3f} (should be ~0)")
print(f"Residual std: {residuals.std():.3f}")

## 📍 7. Comparing KNN to Previous Models

Let's put KNN in context with the models from previous notebooks. We'll compare on both classification (Wine) and regression (Diabetes) tasks, focusing on:
- **Predictive performance** (accuracy/MSE)
- **Training time** (usually negligible for KNN, but prediction time matters)
- **Memory usage** (KNN stores all training data)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.model_selection import cross_validate
import time

# Classification comparison
wine = load_wine()
X, y = wine.data, wine.target

classifiers = {
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(max_iter=1000, random_state=42))
    ]),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM (RBF)': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', SVC(kernel='rbf', random_state=42))
    ]),
    'KNN (k=5)': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', KNeighborsClassifier(n_neighbors=5))
    ])
}

comparison_results = []

for name, clf in classifiers.items():
    start = time.time()
    
    # Cross-validate
    scores = cross_validate(
        clf, X, y, cv=5,
        scoring=['accuracy', 'f1_weighted'],
        return_train_score=True
    )
    
    elapsed = time.time() - start
    
    comparison_results.append({
        'Model': name,
        'Test Accuracy': scores['test_accuracy'].mean(),
        'Test F1': scores['test_f1_weighted'].mean(),
        'Train Accuracy': scores['train_accuracy'].mean(),
        'Fit+Score Time (s)': elapsed
    })

comp_df = pd.DataFrame(comparison_results)
print("Classification Comparison (Wine Dataset):")
print(comp_df.round(4).to_string(index=False))

In [ ]:
# Visualize comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Accuracy
axes[0].barh(comp_df['Model'], comp_df['Test Accuracy'], color='skyblue')
axes[0].set_xlabel('Test Accuracy')
axes[0].set_title('Classification Accuracy')
axes[0].set_xlim(0.8, 1.0)
for i, v in enumerate(comp_df['Test Accuracy']):
    axes[0].text(v + 0.005, i, f'{v:.3f}', va='center')

# Overfitting gap
gap = comp_df['Train Accuracy'] - comp_df['Test Accuracy']
axes[1].barh(comp_df['Model'], gap, color='coral')
axes[1].set_xlabel('Train - Test Accuracy')
axes[1].set_title('Overfitting Gap (smaller is better)')
for i, v in enumerate(gap):
    axes[1].text(v + 0.005, i, f'{v:.3f}', va='center')

# Runtime (log scale)
axes[2].barh(comp_df['Model'], comp_df['Fit+Score Time (s)'], color='lightgreen')
axes[2].set_xlabel('Total Time (s)')
axes[2].set_title('Computation Time (log scale)')
axes[2].set_xscale('log')

plt.tight_layout()
plt.show()

print("\nKNN Characteristics:")
print("- Competitive accuracy on this small dataset")
print("- Zero training time (lazy learner)")
print("- High prediction time (scales with training set size)")
print("- No overfitting gap (memorization vs generalization trade-off is different)")

## ⚠️ Common Pitfalls & Pro Tips

**Critical Mistakes to Avoid:**

- **Forgetting to scale features** → Distance calculations dominated by large-range variables. Always use StandardScaler or MinMaxScaler with KNN.
- **Too small k** → Noise-sensitive, high variance, jagged boundaries. k=1 almost always overfits.
- **Too large k** → Over-smoothing, high bias, misses local structure. Large k approaches the global majority class.
- **High dimensions without dimensionality reduction** → Distance becomes meaningless. Apply PCA or feature selection before KNN when $p > 20$.
- **No built-in feature selection** → KNN uses all features equally. Remove irrelevant features or they will drown out signal.
- **Prediction slow on large n** → $O(n)$ per query naive, $O(\log n)$ with KDTree/BallTree. For $n > 10000$, consider approximate nearest neighbors (ANN) libraries like FAISS or Annoy.
- **Imbalanced classes** → Majority class dominates. Use `weights='distance'` or sample balancing techniques.
- **Categorical features** → Standard distance metrics don't make sense. Use one-hot encoding + Hamming distance, or better, use a different algorithm.
- **Ignoring the 'uniform' vs 'distance' trade-off** → Uniform weights treat far and near neighbors equally. Distance weights often improve performance but add hyperparameter complexity.
- **Using KNN when you need interpretability** → "The prediction is the average of these 5 training points" is less actionable than "If feature X > 3, predict class A" (tree) or "The weight vector is..." (linear model).

## 📝 Exercises

Test your understanding with these progressively challenging exercises:

**Easy:**
1. On the digits dataset (handwritten digits 0-9), use GridSearchCV to find the best combination of k, weights, and metric. What's the optimal configuration? How does it compare to the default k=5, uniform, euclidean?

**Medium:**
2. On the `make_circles` dataset, plot decision boundaries for k=3 vs k=20 with `weights='distance'`. Explain why k=3 creates more "islands" and whether distance weighting helps with the circles pattern.

3. Add 50 noisy random features to the Iris dataset. Compare KNN performance with and without scaling, and with scaling + PCA (retain 95% variance). How much does PCA recover the lost performance?

**Hard:**
4. Implement a simple 1-NN classifier from scratch using only NumPy (no sklearn). It should support Euclidean distance and predict on new data. Compare its predictions to sklearn's KNeighborsClassifier on a 2D toy dataset—are they identical?

**Bonus:**
5. Use KNN for regression on the California housing dataset. Compare it to SVR (from Notebook 9) and RandomForestRegressor. Which handles the long-tail distribution of house prices best? Plot predicted vs actual for all three.

<details>
<summary><strong>💡 Exercise Solutions (Click to expand)</strong></summary>

### Exercise 1: GridSearchCV on Digits
```python
from sklearn.datasets import load_digits

digits = load_digits()
X, y = digits.data, digits.target

param_grid = {
    'knn__n_neighbors': [3, 5, 7, 9, 11],
    'knn__weights': ['uniform', 'distance'],
    'knn__metric': ['euclidean', 'manhattan']
}

grid = GridSearchCV(
    Pipeline([('scaler', StandardScaler()), 
              ('knn', KNeighborsClassifier())]),
    param_grid, cv=5, n_jobs=-1
)
grid.fit(X, y)
print(f"Best: {grid.best_params_}, Score: {grid.best_score_:.4f}")
```

### Exercise 2: Circles with Distance Weighting
```python
X, y = make_circles(n_samples=300, noise=0.1, factor=0.5, random_state=42)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for idx, k in enumerate([3, 20]):
    knn = Pipeline([
        ('scaler', StandardScaler()),
        ('knn', KNeighborsClassifier(n_neighbors=k, weights='distance'))
    ])
    knn.fit(X, y)
    
    # Plot decision boundary (similar to section 2)
    # ... contourf code ...
    axes[idx].set_title(f'k={k}, distance weights')
```

### Exercise 3: Noisy Features + PCA Recovery
```python
from sklearn.decomposition import PCA

iris = load_iris()
X, y = iris.data, iris.target

# Add 50 noise features
np.random.seed(42)
X_noisy = np.hstack([X, np.random.randn(150, 50)])

configs = [
    ('Raw noisy', X_noisy),
    ('Scaled', StandardScaler().fit_transform(X_noisy)),
    ('Scaled + PCA', Pipeline([('scaler', StandardScaler()), 
                               ('pca', PCA(0.95))]).fit_transform(X_noisy))
]

for name, X_transformed in configs:
    scores = cross_val_score(KNeighborsClassifier(5), X_transformed, y, cv=5)
    print(f"{name}: {scores.mean():.3f}")
```

### Exercise 4: Scratch 1-NN Implementation
```python
class Scratch1NN:
    def __init__(self):
        self.X_train = None
        self.y_train = None
    
    def fit(self, X, y):
        self.X_train = X
        self.y_train = y
        return self
    
    def predict(self, X):
        # Compute all pairwise distances
        dists = np.sqrt(((X[:, None, :] - self.X_train[None, :, :]) ** 2).sum(axis=2))
        nearest_idx = dists.argmin(axis=1)
        return self.y_train[nearest_idx]

# Test on 2D data
X, y = make_classification(n_samples=100, n_features=2, n_classes=2, 
                           n_redundant=0, random_state=42)
scratch = Scratch1NN().fit(X, y)
sklearn_knn = KNeighborsClassifier(n_neighbors=1).fit(X, y)

print(f"Predictions match: {np.allclose(scratch.predict(X), sklearn_knn.predict(X))}")
```

### Exercise 5: California Housing Comparison
```python
from sklearn.datasets import fetch_california_housing
from sklearn.svm import SVR

housing = fetch_california_housing()
X, y = housing.data, housing.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

models = {
    'KNN': Pipeline([('scaler', StandardScaler()), 
                     ('knn', KNeighborsRegressor(5))]),
    'SVR': Pipeline([('scaler', StandardScaler()), 
                     ('svr', SVR(kernel='rbf'))]),
    'RF': RandomForestRegressor(n_estimators=100, random_state=42)
}

for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    print(f"{name}: MSE={mean_squared_error(y_test, pred):.4f}, "
          f"R²={r2_score(y_test, pred):.4f}")
```
</details>

## 🎓 Summary – What You Learned Today

**Core Concepts:**
- KNN is a **lazy learner**—no training phase, just storage and lookup
- Predictions are based on **local neighborhoods**, making KNN highly adaptive to local structure but sensitive to noise
- The choice of **k** controls the bias-variance trade-off: small k = complex boundaries (high variance), large k = smooth boundaries (high bias)
- **Distance metrics** fundamentally change what "similar" means—Euclidean for dense continuous data, Manhattan for high dimensions or outliers
- **Weighting schemes** allow closer neighbors to have more influence, often improving performance
- The **curse of dimensionality** makes KNN ineffective in high-dimensional spaces without dimensionality reduction

**Practical Skills:**
- Using Pipeline to ensure scaling happens before distance calculations
- Selecting optimal k via cross-validation and the elbow method
- Visualizing decision boundaries to understand model complexity
- Applying KNN to both classification and regression problems
- Recognizing when KNN is appropriate vs when to use eager learners

**When to Use KNN:**
- Small to medium datasets ($n < 10000$) with low dimensions ($p < 20$)
- When decision boundaries are highly irregular/non-linear
- When interpretability via "similar examples" is valuable
- As a baseline—if KNN works well, the problem has strong local structure

**When NOT to Use KNN:**
- High-dimensional data (curse of dimensionality)
- Very large datasets (slow prediction, high memory)
- When fast real-time prediction is needed (query time scales with data)
- When global patterns dominate local ones
- With mixed data types (categorical + continuous) without careful preprocessing

---

## 🔮 Next Notebook Preview

**Notebook 11: Model Evaluation, Metrics & Cross-Validation Strategies**

We've been using train_test_split and cross_val_score, but it's time to go deeper. You'll master:
- Beyond accuracy: Precision, Recall, F1, ROC-AUC, log-loss, and when to use each
- Stratified K-Fold, Group K-Fold, and Time Series Split for non-iid data
- Nested cross-validation for unbiased hyperparameter tuning evaluation
- Learning curves and validation curves to diagnose bias vs variance
- Statistical significance testing for model comparison
- The bias-variance decomposition and how to measure it

Get ready to stop guessing and start measuring model performance like a pro!